# Can we predict next-month arrears?
Target: `arrears_next_month` (1 = account in arrears in the month after the billing month).

Plan: inspect the data, check for leakage, build features available at billing time, evaluate with a **time-based split** (train on the past, test on the most recent months), compare against simple baselines.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, RocCurveDisplay, PrecisionRecallDisplay
from sklearn.inspection import permutation_importance
pd.set_option('display.width',140)
df=pd.read_csv('data.csv',parse_dates=['billing_month']).sort_values(['meter_ref','billing_month']).reset_index(drop=True)
print(df.shape, df.isna().sum().sum(),'missing;', df.meter_ref.nunique(),'accounts;', df.duplicated(['meter_ref','billing_month']).sum(),'duplicate account-months')
print(df.billing_month.min().date(),'->',df.billing_month.max().date())
df.describe().T

## Target and basic relationships

In [ ]:
print('Arrears rate:', df.arrears_next_month.mean().round(4))
fig,ax=plt.subplots(1,3,figsize=(15,3.5))
df.groupby('billing_month').arrears_next_month.mean().plot(ax=ax[0],title='Arrears rate by month')
df.groupby('late_payments_12m').arrears_next_month.mean().plot.bar(ax=ax[1],title='Rate by late payments (12m)')
df.groupby('tariff').arrears_next_month.mean().plot.bar(ax=ax[2],title='Rate by tariff')
plt.tight_layout(); plt.show()

**The arrears rate is not stable over time**: it rises from ~3% in late 2024 to ~26% by Dec 2025, and the average bill per kWh steps up in Jan 2025 (tariff price rise). Any model trained on the past will face a much higher base rate in the test period.

In [ ]:
m=df.assign(pp=df.bill_eur/df.kwh).groupby('billing_month').agg(arrears_rate=('arrears_next_month','mean'),eur_per_kwh=('pp','mean'),late_pay=('late_payments_12m','mean'))
fig,ax=plt.subplots(1,3,figsize=(15,3)); [m[c].plot(ax=a,title=c) for c,a in zip(m,ax)]; plt.tight_layout(); plt.show()

## Leakage check: `final_notice_sent`
The dictionary says it is set if a final notice was sent *in the billing cycle that followed* the billing month. That is after the prediction point, and notices are presumably a consequence of arrears. Let's see how strongly it tracks the target.

In [ ]:
print(df.groupby('final_notice_sent').arrears_next_month.agg(['mean','size']))
print('AUC of final_notice_sent alone:', round(roc_auc_score(df.arrears_next_month, df.final_notice_sent),3))

It is almost a restatement of the target and would not be known when we need to make the prediction, so **it is excluded from the real model**. It is only used below to show how much it would inflate results.

## Features
Only information known at the end of the billing month: current-row fields plus each account's own *past* history (lags / expanding means via `shift`, so no future rows leak in).

In [ ]:
g=df.groupby('meter_ref')
df['price_per_kwh']=df.bill_eur/df.kwh
df['kwh_per_occ']=df.kwh/df.occupants
df['bill_per_occ']=df.bill_eur/df.occupants
df['tenure']=g.cumcount()                      # months observed so far (left-censored at 2023-07)
for col in ['kwh','bill_eur','late_payments_12m']:
    df[f'{col}_prev']=g[col].shift(1)
    df[f'{col}_chg']=df[col]-df[f'{col}_prev']
df['kwh_mean_past']=g['kwh'].transform(lambda s:s.shift(1).expanding().mean())
df['kwh_vs_own']=df.kwh/df.kwh_mean_past
df['month']=df.billing_month.dt.month
df=pd.get_dummies(df,columns=['tariff'],dtype=int)
target='arrears_next_month'
feats=[c for c in df.columns if c not in ['meter_ref','billing_month',target,'final_notice_sent']]
print(feats)

## Time-based split
Train on months before 2025-07, test on 2025-07 → 2025-12. A random split would let the model see the same account's neighbouring months and the future. (Accounts do recur across the split, as they would in production.)

In [ ]:
cut=pd.Timestamp('2025-07-01')
tr,te=df[df.billing_month<cut],df[df.billing_month>=cut]
ytr,yte=tr[target],te[target]
print(len(tr),len(te),'| arrears rate train/test',ytr.mean().round(3),yte.mean().round(3))
def ev(name,p):
    return dict(model=name,mean_pred=p.mean(),ROC_AUC=roc_auc_score(yte,p),PR_AUC=average_precision_score(yte,p),Brier=brier_score_loss(yte,p))
preds={}
preds['late_payments only']=LogisticRegression().fit(tr[['late_payments_12m']],ytr).predict_proba(te[['late_payments_12m']])[:,1]
med=tr[feats].median()
lr=make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000)).fit(tr[feats].fillna(med),ytr)
preds['logistic']=lr.predict_proba(te[feats].fillna(med))[:,1]
hgb=HistGradientBoostingClassifier(max_depth=4,learning_rate=0.05,max_iter=300,l2_regularization=1.0,random_state=0).fit(tr[feats],ytr)
preds['grad. boosting']=hgb.predict_proba(te[feats])[:,1]
lf=feats+['final_notice_sent']   # leaky, for comparison only
preds['(leaky) boosting + final_notice']=HistGradientBoostingClassifier(max_depth=4,learning_rate=0.05,max_iter=300,random_state=0).fit(tr[lf],ytr).predict_proba(te[lf])[:,1]
res=[dict(model='base rate (no skill)',mean_pred=ytr.mean(),ROC_AUC=0.5,PR_AUC=yte.mean(),Brier=brier_score_loss(yte,np.full(len(yte),ytr.mean())))]
res+=[ev(k,v) for k,v in preds.items()]
print('actual test arrears rate:',yte.mean().round(3))
pd.DataFrame(res).set_index('model').round(3)

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4))
for k,v in preds.items():
    RocCurveDisplay.from_predictions(yte,v,name=k,ax=ax[0]); PrecisionRecallDisplay.from_predictions(yte,v,name=k,ax=ax[1])
ax[1].axhline(yte.mean(),ls=':',c='gray'); plt.tight_layout(); plt.show()

## What drives the prediction? (permutation importance on the test set, honest model)

In [ ]:
pi=permutation_importance(hgb,te[feats],yte,scoring='roc_auc',n_repeats=5,random_state=0)
pd.Series(pi.importances_mean,feats).sort_values().tail(10).plot.barh(title='Drop in test AUC when shuffled'); plt.show()

## Practical value: who would we flag?

In [ ]:
p=preds['grad. boosting']; t=pd.DataFrame({'y':yte.values,'p':p})
t['decile']=pd.qcut(t.p.rank(method='first'),10,labels=range(1,11))
display(t.groupby('decile',observed=True).agg(accounts=('y','size'),arrears_rate=('y','mean'),mean_pred=('p','mean')).round(3))
for k in (0.1,0.2):
    top=t.nlargest(int(k*len(t)),'p'); print(f'Flag top {int(k*100)}%: precision {top.y.mean():.2f}, recall {top.y.sum()/t.y.sum():.2f}, lift {top.y.mean()/t.y.mean():.1f}x')

## Caveats
- `final_notice_sent` is excluded because it post-dates the prediction point; including it gives a deceptively high score.
- Features are limited (no payment amounts, income, payment-method data); history before 2023-07 is unobserved, so `tenure` is censored.
- The test window is six months from one account population; check calibration and drift before deploying, and choose a flag threshold from the cost of a missed arrears vs. a needless contact.